# 🚖 Caso Práctico 3 — RapidX: Pipeline JSON → Parquet → BI

## 🌍 Contexto de negocio

**RapidX** es una plataforma de movilidad urbana bajo demanda (estilo Uber) que opera en cinco ciudades españolas. Sus sistemas operacionales generan **un fichero JSON por ciudad y día** con todos los viajes (completados, cancelados, en curso).

Encargo del equipo de **Business Intelligence (BI)**:

> *"Necesitamos una vista limpia y enriquecida de los viajes para responder preguntas de negocio desde SQL. Los JSONs operacionales tienen ruido, campos internos (`_sys_version`, `_batch_id`) y columnas que no necesitamos. Queremos un Parquet final con solo las columnas relevantes, particionado por ciudad, consultable con Spark SQL."*

## 🎯 Pipeline

1. Generar **7 ficheros JSON sintéticos** (Madrid×2, Barcelona, Sevilla, Valencia, Bilbao, multi-ciudad).
2. **Leer** todos los JSON con Spark.
3. **Enriquecer** con funciones nativas (`when`, `to_timestamp`, `hour`, `round`...).
4. **Seleccionar** solo columnas BI (descartar `_sys_version`, `_batch_id`, timestamps raw).
5. **Escribir** Parquet particionado por `ciudad`.
6. **Exponer** como vista SQL `viajes_rapidx` y resolver 7 consultas BI.

---

## 🔧 Parte 1 — Inicialización del entorno

> Ejecuta esta celda **antes que cualquier otra**.

In [1]:
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.log4j.{Level, Logger}
Logger.getLogger("org").setLevel(Level.ERROR)
Logger.getLogger("akka").setLevel(Level.ERROR)

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._

import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
import java.io.File

val spark = SparkSession.builder()
  .appName("RapidX_BI_Pipeline")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .config("spark.sql.shuffle.partitions", "4")
  .getOrCreate()

import spark.implicits._

spark.sparkContext.setLogLevel("ERROR")

println(s"✅ Spark ${spark.version} listo — RapidX BI Pipeline")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/06 13:51:33 INFO SparkContext: Running Spark version 4.1.1
26/05/06 13:51:33 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/05/06 13:51:33 INFO SparkContext: Java version 17.0.18+8
26/05/06 13:51:35 INFO ResourceUtils: ==============================================================
26/05/06 13:51:35 INFO ResourceUtils: No custom resources configured for spark.driver.
26/05/06 13:51:35 INFO ResourceUtils: ==============================================================
26/05/06 13:51:35 INFO SparkContext: Submitted application: RapidX_BI_Pipeline
26/05/06 13:51:35 INFO SecurityManager: Changing view acls to: gre
26/05/06 13:51:35 INFO SecurityManager: Changing modify acls to: gre
26/05/06 13:51:35 INFO SecurityManager: Changing view acls groups to: gre
26/05/06 13:51:35 INFO SecurityManager: Changing modify acls groups to: gre
26/05/06 13:51:35 INFO SecurityManager: SecurityManager: authentication

✅ Spark 4.1.1 listo — RapidX BI Pipeline


import $ivy.$
import $ivy.$
import org.apache.log4j.{Level, Logger}
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
import java.io.File
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@d830ed2
import spark.implicits._

---

## 🔧 Parte 2 — Estructura de carpetas

Caso autocontenido: las carpetas se crean **dentro de la carpeta del notebook** para que el caso sea portable.

In [2]:
// Rutas relativas a la carpeta del notebook → caso autocontenido y portable.
val rutaBase   = "."
val rutaJSON   = s"$rutaBase/json_raw"
val rutaSalida = s"$rutaBase/salida"

List(rutaJSON, rutaSalida).foreach { c =>
  Files.createDirectories(Paths.get(c))
  println(s"  📁 $c")
}

println("\n✅ Estructura de carpetas lista")

  📁 ./json_raw
  📁 ./salida

✅ Estructura de carpetas lista


rutaBase: String = "."
rutaJSON: String = "./json_raw"
rutaSalida: String = "./salida"

---

## 📄 Parte 3 — Generación de los 7 ficheros JSON sintéticos

Cada fichero representa **un día de operaciones en una ciudad**. Incluyen campos internos (`_sys_version`, `_batch_id`) que el equipo BI **no necesita** y descartaremos en la Parte 6.

### Fichero 1 — Madrid día 1 (10 viajes)

In [3]:
val jsonMadrid1 =
"""[
  {"viaje_id":"VJ-M-0001","timestamp_inicio":"2024-11-01T07:12:33","timestamp_fin":"2024-11-01T07:38:55","ciudad":"Madrid","zona_origen":"Chamberí","zona_destino":"Retiro","distancia_km":4.8,"duracion_min":26,"tarifa_base":6.40,"propina":1.20,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-441","pasajero_id":"PAX-9921","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0002","timestamp_inicio":"2024-11-01T08:05:10","timestamp_fin":"2024-11-01T08:29:40","ciudad":"Madrid","zona_origen":"Sol","zona_destino":"Moncloa","distancia_km":3.2,"duracion_min":24,"tarifa_base":5.10,"propina":0.0,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":4,"conductor_id":"DRV-207","pasajero_id":"PAX-3341","tipo_vehiculo":"UberX","surge_multiplier":1.2,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0003","timestamp_inicio":"2024-11-01T08:45:00","timestamp_fin":null,"ciudad":"Madrid","zona_origen":"Vallecas","zona_destino":"Barajas","distancia_km":0.0,"duracion_min":0,"tarifa_base":0.0,"propina":0.0,"metodo_pago":"tarjeta","estado":"cancelado","calificacion_pasajero":null,"calificacion_conductor":null,"conductor_id":"DRV-119","pasajero_id":"PAX-5512","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0004","timestamp_inicio":"2024-11-01T09:30:15","timestamp_fin":"2024-11-01T10:05:50","ciudad":"Madrid","zona_origen":"Salamanca","zona_destino":"Tetuán","distancia_km":6.1,"duracion_min":35,"tarifa_base":9.20,"propina":2.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-441","pasajero_id":"PAX-7720","tipo_vehiculo":"UberComfort","surge_multiplier":1.5,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0005","timestamp_inicio":"2024-11-01T11:00:00","timestamp_fin":"2024-11-01T11:18:30","ciudad":"Madrid","zona_origen":"Arganzuela","zona_destino":"Centro","distancia_km":2.9,"duracion_min":18,"tarifa_base":4.80,"propina":0.50,"metodo_pago":"app_wallet","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":5,"conductor_id":"DRV-332","pasajero_id":"PAX-1102","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0006","timestamp_inicio":"2024-11-01T12:15:44","timestamp_fin":"2024-11-01T12:55:10","ciudad":"Madrid","zona_origen":"Barajas","zona_destino":"Chamartín","distancia_km":11.3,"duracion_min":39,"tarifa_base":14.50,"propina":3.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":4,"conductor_id":"DRV-207","pasajero_id":"PAX-8841","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0007","timestamp_inicio":"2024-11-01T14:00:00","timestamp_fin":"2024-11-01T14:22:00","ciudad":"Madrid","zona_origen":"Hortaleza","zona_destino":"Retiro","distancia_km":7.7,"duracion_min":22,"tarifa_base":10.10,"propina":0.0,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":3,"calificacion_conductor":5,"conductor_id":"DRV-885","pasajero_id":"PAX-2230","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0008","timestamp_inicio":"2024-11-01T17:30:00","timestamp_fin":"2024-11-01T18:10:20","ciudad":"Madrid","zona_origen":"Centro","zona_destino":"Vallecas","distancia_km":8.9,"duracion_min":40,"tarifa_base":12.00,"propina":2.50,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-119","pasajero_id":"PAX-4410","tipo_vehiculo":"UberComfort","surge_multiplier":2.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0009","timestamp_inicio":"2024-11-01T19:45:00","timestamp_fin":null,"ciudad":"Madrid","zona_origen":"Moncloa","zona_destino":"Sol","distancia_km":0.0,"duracion_min":0,"tarifa_base":0.0,"propina":0.0,"metodo_pago":"app_wallet","estado":"cancelado","calificacion_pasajero":null,"calificacion_conductor":null,"conductor_id":"DRV-332","pasajero_id":"PAX-6631","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0010","timestamp_inicio":"2024-11-01T22:10:05","timestamp_fin":"2024-11-01T22:45:30","ciudad":"Madrid","zona_origen":"Retiro","zona_destino":"Salamanca","distancia_km":3.5,"duracion_min":35,"tarifa_base":7.80,"propina":1.50,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":4,"conductor_id":"DRV-441","pasajero_id":"PAX-9921","tipo_vehiculo":"UberX","surge_multiplier":1.8,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"}
]"""

val ruta1 = s"$rutaJSON/viajes_madrid_2024-11-01.json"
Files.write(Paths.get(ruta1), jsonMadrid1.getBytes(StandardCharsets.UTF_8))
println(s"✅ Creado: $ruta1")

✅ Creado: ./json_raw/viajes_madrid_2024-11-01.json


jsonMadrid1: String = """[
  {"viaje_id":"VJ-M-0001","timestamp_inicio":"2024-11-01T07:12:33","timestamp_fin":"2024-11-01T07:38:55","ciudad":"Madrid","zona_origen":"Chamberí","zona_destino":"Retiro","distancia_km":4.8,"duracion_min":26,"tarifa_base":6.40,"propina":1.20,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-441","pasajero_id":"PAX-9921","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-MAD"},
  {"viaje_id":"VJ-M-0002","timestamp_inicio":"2024-11-01T08:05:10","timestamp_fin":"2024-11-01T08:29:40","ciudad":"Madrid","zona_origen":"Sol","zona_destino":"Moncloa","distancia_km":3.2,"duracion_min":24,"tarifa_base":5.10,"propina":0.0,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":4,"conductor_id":"DRV-207","pasajero_id":"PAX-3341","tipo_vehiculo":"UberX","surge_multiplier":1.2,"_sys_version":"v3.2","_batch_id":"BAT-20241

### Fichero 2 — Madrid día 2 (5 viajes)

In [4]:
val jsonMadrid2 =
"""[
  {"viaje_id":"VJ-M-0011","timestamp_inicio":"2024-11-02T06:55:00","timestamp_fin":"2024-11-02T07:25:10","ciudad":"Madrid","zona_origen":"Tetuán","zona_destino":"Chamartín","distancia_km":4.1,"duracion_min":30,"tarifa_base":6.80,"propina":0.0,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":4,"conductor_id":"DRV-207","pasajero_id":"PAX-1102","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241102-MAD"},
  {"viaje_id":"VJ-M-0012","timestamp_inicio":"2024-11-02T08:20:00","timestamp_fin":"2024-11-02T09:00:45","ciudad":"Madrid","zona_origen":"Barajas","zona_destino":"Centro","distancia_km":14.2,"duracion_min":40,"tarifa_base":18.50,"propina":4.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-119","pasajero_id":"PAX-7720","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241102-MAD"},
  {"viaje_id":"VJ-M-0013","timestamp_inicio":"2024-11-02T10:00:00","timestamp_fin":"2024-11-02T10:19:30","ciudad":"Madrid","zona_origen":"Retiro","zona_destino":"Arganzuela","distancia_km":3.3,"duracion_min":19,"tarifa_base":5.60,"propina":1.00,"metodo_pago":"app_wallet","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-885","pasajero_id":"PAX-3341","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241102-MAD"},
  {"viaje_id":"VJ-M-0014","timestamp_inicio":"2024-11-02T13:30:00","timestamp_fin":"2024-11-02T14:10:00","ciudad":"Madrid","zona_origen":"Salamanca","zona_destino":"Vallecas","distancia_km":9.2,"duracion_min":40,"tarifa_base":13.40,"propina":2.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":5,"conductor_id":"DRV-441","pasajero_id":"PAX-5512","tipo_vehiculo":"UberComfort","surge_multiplier":1.3,"_sys_version":"v3.2","_batch_id":"BAT-20241102-MAD"},
  {"viaje_id":"VJ-M-0015","timestamp_inicio":"2024-11-02T18:45:00","timestamp_fin":"2024-11-02T19:30:00","ciudad":"Madrid","zona_origen":"Centro","zona_destino":"Moncloa","distancia_km":5.5,"duracion_min":45,"tarifa_base":9.90,"propina":0.0,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":3,"calificacion_conductor":4,"conductor_id":"DRV-332","pasajero_id":"PAX-8841","tipo_vehiculo":"UberX","surge_multiplier":1.9,"_sys_version":"v3.2","_batch_id":"BAT-20241102-MAD"}
]"""

val ruta2 = s"$rutaJSON/viajes_madrid_2024-11-02.json"
Files.write(Paths.get(ruta2), jsonMadrid2.getBytes(StandardCharsets.UTF_8))
println(s"✅ Creado: $ruta2")

✅ Creado: ./json_raw/viajes_madrid_2024-11-02.json


jsonMadrid2: String = """[
  {"viaje_id":"VJ-M-0011","timestamp_inicio":"2024-11-02T06:55:00","timestamp_fin":"2024-11-02T07:25:10","ciudad":"Madrid","zona_origen":"Tetuán","zona_destino":"Chamartín","distancia_km":4.1,"duracion_min":30,"tarifa_base":6.80,"propina":0.0,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":4,"conductor_id":"DRV-207","pasajero_id":"PAX-1102","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241102-MAD"},
  {"viaje_id":"VJ-M-0012","timestamp_inicio":"2024-11-02T08:20:00","timestamp_fin":"2024-11-02T09:00:45","ciudad":"Madrid","zona_origen":"Barajas","zona_destino":"Centro","distancia_km":14.2,"duracion_min":40,"tarifa_base":18.50,"propina":4.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-119","pasajero_id":"PAX-7720","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT

### Fichero 3 — Barcelona (7 viajes)

In [5]:
val jsonBarcelona =
"""[
  {"viaje_id":"VJ-B-0001","timestamp_inicio":"2024-11-01T07:30:00","timestamp_fin":"2024-11-01T08:00:15","ciudad":"Barcelona","zona_origen":"Gràcia","zona_destino":"Eixample","distancia_km":3.1,"duracion_min":30,"tarifa_base":5.80,"propina":1.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-610","pasajero_id":"PAX-2201","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BCN"},
  {"viaje_id":"VJ-B-0002","timestamp_inicio":"2024-11-01T09:00:00","timestamp_fin":"2024-11-01T09:35:20","ciudad":"Barcelona","zona_origen":"Sants","zona_destino":"Barceloneta","distancia_km":5.7,"duracion_min":35,"tarifa_base":8.30,"propina":0.0,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":3,"conductor_id":"DRV-730","pasajero_id":"PAX-4410","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BCN"},
  {"viaje_id":"VJ-B-0003","timestamp_inicio":"2024-11-01T10:10:00","timestamp_fin":null,"ciudad":"Barcelona","zona_origen":"Sagrada Familia","zona_destino":"Gràcia","distancia_km":0.0,"duracion_min":0,"tarifa_base":0.0,"propina":0.0,"metodo_pago":"app_wallet","estado":"cancelado","calificacion_pasajero":null,"calificacion_conductor":null,"conductor_id":"DRV-610","pasajero_id":"PAX-6631","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BCN"},
  {"viaje_id":"VJ-B-0004","timestamp_inicio":"2024-11-01T12:00:00","timestamp_fin":"2024-11-01T12:50:00","ciudad":"Barcelona","zona_origen":"Eixample","zona_destino":"Aeropuerto BCN","distancia_km":15.8,"duracion_min":50,"tarifa_base":22.00,"propina":5.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-850","pasajero_id":"PAX-9921","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BCN"},
  {"viaje_id":"VJ-B-0005","timestamp_inicio":"2024-11-01T14:30:00","timestamp_fin":"2024-11-01T14:55:10","ciudad":"Barcelona","zona_origen":"Barceloneta","zona_destino":"Poblenou","distancia_km":4.2,"duracion_min":25,"tarifa_base":6.90,"propina":1.50,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":5,"conductor_id":"DRV-730","pasajero_id":"PAX-2201","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BCN"},
  {"viaje_id":"VJ-B-0006","timestamp_inicio":"2024-11-01T18:00:00","timestamp_fin":"2024-11-01T18:40:00","ciudad":"Barcelona","zona_origen":"Sants","zona_destino":"Sagrada Familia","distancia_km":6.3,"duracion_min":40,"tarifa_base":10.20,"propina":2.00,"metodo_pago":"app_wallet","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-610","pasajero_id":"PAX-3341","tipo_vehiculo":"UberComfort","surge_multiplier":1.6,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BCN"},
  {"viaje_id":"VJ-B-0007","timestamp_inicio":"2024-11-01T20:15:00","timestamp_fin":"2024-11-01T20:45:30","ciudad":"Barcelona","zona_origen":"Poblenou","zona_destino":"Eixample","distancia_km":5.0,"duracion_min":30,"tarifa_base":8.50,"propina":0.0,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":3,"calificacion_conductor":4,"conductor_id":"DRV-850","pasajero_id":"PAX-7720","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BCN"}
]"""

val ruta3 = s"$rutaJSON/viajes_barcelona_2024-11-01.json"
Files.write(Paths.get(ruta3), jsonBarcelona.getBytes(StandardCharsets.UTF_8))
println(s"✅ Creado: $ruta3")

✅ Creado: ./json_raw/viajes_barcelona_2024-11-01.json


jsonBarcelona: String = """[
  {"viaje_id":"VJ-B-0001","timestamp_inicio":"2024-11-01T07:30:00","timestamp_fin":"2024-11-01T08:00:15","ciudad":"Barcelona","zona_origen":"Gràcia","zona_destino":"Eixample","distancia_km":3.1,"duracion_min":30,"tarifa_base":5.80,"propina":1.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-610","pasajero_id":"PAX-2201","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BCN"},
  {"viaje_id":"VJ-B-0002","timestamp_inicio":"2024-11-01T09:00:00","timestamp_fin":"2024-11-01T09:35:20","ciudad":"Barcelona","zona_origen":"Sants","zona_destino":"Barceloneta","distancia_km":5.7,"duracion_min":35,"tarifa_base":8.30,"propina":0.0,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":3,"conductor_id":"DRV-730","pasajero_id":"PAX-4410","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_

### Fichero 4 — Sevilla (5 viajes)

In [6]:
val jsonSevilla =
"""[
  {"viaje_id":"VJ-S-0001","timestamp_inicio":"2024-11-01T08:00:00","timestamp_fin":"2024-11-01T08:22:00","ciudad":"Sevilla","zona_origen":"Triana","zona_destino":"Centro","distancia_km":3.5,"duracion_min":22,"tarifa_base":5.20,"propina":0.50,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-561","pasajero_id":"PAX-1102","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-SVQ"},
  {"viaje_id":"VJ-S-0002","timestamp_inicio":"2024-11-01T09:30:00","timestamp_fin":"2024-11-01T10:05:00","ciudad":"Sevilla","zona_origen":"Macarena","zona_destino":"Aeropuerto SVQ","distancia_km":9.8,"duracion_min":35,"tarifa_base":12.50,"propina":2.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":5,"conductor_id":"DRV-623","pasajero_id":"PAX-4410","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-SVQ"},
  {"viaje_id":"VJ-S-0003","timestamp_inicio":"2024-11-01T11:00:00","timestamp_fin":null,"ciudad":"Sevilla","zona_origen":"Nervión","zona_destino":"Triana","distancia_km":0.0,"duracion_min":0,"tarifa_base":0.0,"propina":0.0,"metodo_pago":"app_wallet","estado":"cancelado","calificacion_pasajero":null,"calificacion_conductor":null,"conductor_id":"DRV-561","pasajero_id":"PAX-8841","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-SVQ"},
  {"viaje_id":"VJ-S-0004","timestamp_inicio":"2024-11-01T13:15:00","timestamp_fin":"2024-11-01T13:40:00","ciudad":"Sevilla","zona_origen":"Centro","zona_destino":"Nervión","distancia_km":4.2,"duracion_min":25,"tarifa_base":6.30,"propina":1.00,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":4,"conductor_id":"DRV-623","pasajero_id":"PAX-2230","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-SVQ"},
  {"viaje_id":"VJ-S-0005","timestamp_inicio":"2024-11-01T17:00:00","timestamp_fin":"2024-11-01T17:28:00","ciudad":"Sevilla","zona_origen":"Triana","zona_destino":"Macarena","distancia_km":5.1,"duracion_min":28,"tarifa_base":7.80,"propina":0.0,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-561","pasajero_id":"PAX-6631","tipo_vehiculo":"UberComfort","surge_multiplier":1.4,"_sys_version":"v3.2","_batch_id":"BAT-20241101-SVQ"}
]"""

val ruta4 = s"$rutaJSON/viajes_sevilla_2024-11-01.json"
Files.write(Paths.get(ruta4), jsonSevilla.getBytes(StandardCharsets.UTF_8))
println(s"✅ Creado: $ruta4")

✅ Creado: ./json_raw/viajes_sevilla_2024-11-01.json


jsonSevilla: String = """[
  {"viaje_id":"VJ-S-0001","timestamp_inicio":"2024-11-01T08:00:00","timestamp_fin":"2024-11-01T08:22:00","ciudad":"Sevilla","zona_origen":"Triana","zona_destino":"Centro","distancia_km":3.5,"duracion_min":22,"tarifa_base":5.20,"propina":0.50,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-561","pasajero_id":"PAX-1102","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-SVQ"},
  {"viaje_id":"VJ-S-0002","timestamp_inicio":"2024-11-01T09:30:00","timestamp_fin":"2024-11-01T10:05:00","ciudad":"Sevilla","zona_origen":"Macarena","zona_destino":"Aeropuerto SVQ","distancia_km":9.8,"duracion_min":35,"tarifa_base":12.50,"propina":2.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":5,"conductor_id":"DRV-623","pasajero_id":"PAX-4410","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_

### Fichero 5 — Valencia (6 viajes)

In [7]:
val jsonValencia =
"""[
  {"viaje_id":"VJ-V-0001","timestamp_inicio":"2024-11-01T07:00:00","timestamp_fin":"2024-11-01T07:30:00","ciudad":"Valencia","zona_origen":"Ruzafa","zona_destino":"Ciudad de las Artes","distancia_km":4.5,"duracion_min":30,"tarifa_base":7.10,"propina":1.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-712","pasajero_id":"PAX-5512","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-VLC"},
  {"viaje_id":"VJ-V-0002","timestamp_inicio":"2024-11-01T08:30:00","timestamp_fin":"2024-11-01T09:10:00","ciudad":"Valencia","zona_origen":"Aeropuerto VLC","zona_destino":"Centro","distancia_km":10.2,"duracion_min":40,"tarifa_base":14.00,"propina":3.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":5,"conductor_id":"DRV-803","pasajero_id":"PAX-9921","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-VLC"},
  {"viaje_id":"VJ-V-0003","timestamp_inicio":"2024-11-01T10:00:00","timestamp_fin":"2024-11-01T10:20:00","ciudad":"Valencia","zona_origen":"Benimaclet","zona_destino":"Ruzafa","distancia_km":5.8,"duracion_min":20,"tarifa_base":8.20,"propina":0.0,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":3,"calificacion_conductor":4,"conductor_id":"DRV-712","pasajero_id":"PAX-3341","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-VLC"},
  {"viaje_id":"VJ-V-0004","timestamp_inicio":"2024-11-01T12:00:00","timestamp_fin":null,"ciudad":"Valencia","zona_origen":"Centro","zona_destino":"Puerto","distancia_km":0.0,"duracion_min":0,"tarifa_base":0.0,"propina":0.0,"metodo_pago":"app_wallet","estado":"cancelado","calificacion_pasajero":null,"calificacion_conductor":null,"conductor_id":"DRV-803","pasajero_id":"PAX-2201","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-VLC"},
  {"viaje_id":"VJ-V-0005","timestamp_inicio":"2024-11-01T15:30:00","timestamp_fin":"2024-11-01T15:55:00","ciudad":"Valencia","zona_origen":"Ciudad de las Artes","zona_destino":"Benimaclet","distancia_km":6.9,"duracion_min":25,"tarifa_base":9.50,"propina":2.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-712","pasajero_id":"PAX-7720","tipo_vehiculo":"UberComfort","surge_multiplier":1.2,"_sys_version":"v3.2","_batch_id":"BAT-20241101-VLC"},
  {"viaje_id":"VJ-V-0006","timestamp_inicio":"2024-11-01T19:00:00","timestamp_fin":"2024-11-01T19:35:00","ciudad":"Valencia","zona_origen":"Puerto","zona_destino":"Centro","distancia_km":4.0,"duracion_min":35,"tarifa_base":7.60,"propina":0.0,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":4,"conductor_id":"DRV-803","pasajero_id":"PAX-4410","tipo_vehiculo":"UberX","surge_multiplier":1.7,"_sys_version":"v3.2","_batch_id":"BAT-20241101-VLC"}
]"""

val ruta5 = s"$rutaJSON/viajes_valencia_2024-11-01.json"
Files.write(Paths.get(ruta5), jsonValencia.getBytes(StandardCharsets.UTF_8))
println(s"✅ Creado: $ruta5")

✅ Creado: ./json_raw/viajes_valencia_2024-11-01.json


jsonValencia: String = """[
  {"viaje_id":"VJ-V-0001","timestamp_inicio":"2024-11-01T07:00:00","timestamp_fin":"2024-11-01T07:30:00","ciudad":"Valencia","zona_origen":"Ruzafa","zona_destino":"Ciudad de las Artes","distancia_km":4.5,"duracion_min":30,"tarifa_base":7.10,"propina":1.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-712","pasajero_id":"PAX-5512","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-VLC"},
  {"viaje_id":"VJ-V-0002","timestamp_inicio":"2024-11-01T08:30:00","timestamp_fin":"2024-11-01T09:10:00","ciudad":"Valencia","zona_origen":"Aeropuerto VLC","zona_destino":"Centro","distancia_km":10.2,"duracion_min":40,"tarifa_base":14.00,"propina":3.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":5,"conductor_id":"DRV-803","pasajero_id":"PAX-9921","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":

### Fichero 6 — Bilbao (5 viajes)

In [8]:
val jsonBilbao =
"""[
  {"viaje_id":"VJ-BI-0001","timestamp_inicio":"2024-11-01T07:45:00","timestamp_fin":"2024-11-01T08:10:00","ciudad":"Bilbao","zona_origen":"Casco Viejo","zona_destino":"Guggenheim","distancia_km":2.8,"duracion_min":25,"tarifa_base":5.50,"propina":1.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-910","pasajero_id":"PAX-1102","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BIO"},
  {"viaje_id":"VJ-BI-0002","timestamp_inicio":"2024-11-01T09:00:00","timestamp_fin":"2024-11-01T09:45:00","ciudad":"Bilbao","zona_origen":"Aeropuerto BIO","zona_destino":"Centro","distancia_km":11.5,"duracion_min":45,"tarifa_base":15.80,"propina":2.50,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":5,"conductor_id":"DRV-975","pasajero_id":"PAX-8841","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BIO"},
  {"viaje_id":"VJ-BI-0003","timestamp_inicio":"2024-11-01T11:00:00","timestamp_fin":"2024-11-01T11:22:00","ciudad":"Bilbao","zona_origen":"Guggenheim","zona_destino":"Indautxu","distancia_km":3.3,"duracion_min":22,"tarifa_base":5.90,"propina":0.0,"metodo_pago":"app_wallet","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":4,"conductor_id":"DRV-910","pasajero_id":"PAX-6631","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BIO"},
  {"viaje_id":"VJ-BI-0004","timestamp_inicio":"2024-11-01T14:00:00","timestamp_fin":"2024-11-01T14:35:00","ciudad":"Bilbao","zona_origen":"Indautxu","zona_destino":"Casco Viejo","distancia_km":4.8,"duracion_min":35,"tarifa_base":8.10,"propina":1.50,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-975","pasajero_id":"PAX-2230","tipo_vehiculo":"UberComfort","surge_multiplier":1.3,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BIO"},
  {"viaje_id":"VJ-BI-0005","timestamp_inicio":"2024-11-01T17:30:00","timestamp_fin":null,"ciudad":"Bilbao","zona_origen":"Centro","zona_destino":"Aeropuerto BIO","distancia_km":0.0,"duracion_min":0,"tarifa_base":0.0,"propina":0.0,"metodo_pago":"tarjeta","estado":"cancelado","calificacion_pasajero":null,"calificacion_conductor":null,"conductor_id":"DRV-910","pasajero_id":"PAX-3341","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BIO"}
]"""

val ruta6 = s"$rutaJSON/viajes_bilbao_2024-11-01.json"
Files.write(Paths.get(ruta6), jsonBilbao.getBytes(StandardCharsets.UTF_8))
println(s"✅ Creado: $ruta6")

✅ Creado: ./json_raw/viajes_bilbao_2024-11-01.json


jsonBilbao: String = """[
  {"viaje_id":"VJ-BI-0001","timestamp_inicio":"2024-11-01T07:45:00","timestamp_fin":"2024-11-01T08:10:00","ciudad":"Bilbao","zona_origen":"Casco Viejo","zona_destino":"Guggenheim","distancia_km":2.8,"duracion_min":25,"tarifa_base":5.50,"propina":1.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-910","pasajero_id":"PAX-1102","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.2","_batch_id":"BAT-20241101-BIO"},
  {"viaje_id":"VJ-BI-0002","timestamp_inicio":"2024-11-01T09:00:00","timestamp_fin":"2024-11-01T09:45:00","ciudad":"Bilbao","zona_origen":"Aeropuerto BIO","zona_destino":"Centro","distancia_km":11.5,"duracion_min":45,"tarifa_base":15.80,"propina":2.50,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":5,"conductor_id":"DRV-975","pasajero_id":"PAX-8841","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.2","

### Fichero 7 — Multi-ciudad día 3 (8 viajes)

Lote de resincronización del sistema: registros de las 5 ciudades en un único fichero.

In [9]:
val jsonMultiCiudad =
"""[
  {"viaje_id":"VJ-M-0020","timestamp_inicio":"2024-11-03T08:00:00","timestamp_fin":"2024-11-03T08:30:00","ciudad":"Madrid","zona_origen":"Sol","zona_destino":"Retiro","distancia_km":3.8,"duracion_min":30,"tarifa_base":6.20,"propina":1.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-441","pasajero_id":"PAX-2201","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.3","_batch_id":"BAT-20241103-ALL"},
  {"viaje_id":"VJ-B-0010","timestamp_inicio":"2024-11-03T09:15:00","timestamp_fin":"2024-11-03T09:50:00","ciudad":"Barcelona","zona_origen":"Eixample","zona_destino":"Sants","distancia_km":5.2,"duracion_min":35,"tarifa_base":8.80,"propina":0.0,"metodo_pago":"app_wallet","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":4,"conductor_id":"DRV-610","pasajero_id":"PAX-4410","tipo_vehiculo":"UberX","surge_multiplier":1.1,"_sys_version":"v3.3","_batch_id":"BAT-20241103-ALL"},
  {"viaje_id":"VJ-S-0010","timestamp_inicio":"2024-11-03T10:00:00","timestamp_fin":"2024-11-03T10:25:00","ciudad":"Sevilla","zona_origen":"Nervión","zona_destino":"Centro","distancia_km":4.0,"duracion_min":25,"tarifa_base":6.50,"propina":0.50,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-623","pasajero_id":"PAX-7720","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.3","_batch_id":"BAT-20241103-ALL"},
  {"viaje_id":"VJ-V-0010","timestamp_inicio":"2024-11-03T11:00:00","timestamp_fin":"2024-11-03T11:40:00","ciudad":"Valencia","zona_origen":"Centro","zona_destino":"Aeropuerto VLC","distancia_km":10.5,"duracion_min":40,"tarifa_base":14.80,"propina":3.50,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-803","pasajero_id":"PAX-9921","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.3","_batch_id":"BAT-20241103-ALL"},
  {"viaje_id":"VJ-BI-0010","timestamp_inicio":"2024-11-03T12:00:00","timestamp_fin":null,"ciudad":"Bilbao","zona_origen":"Guggenheim","zona_destino":"Casco Viejo","distancia_km":0.0,"duracion_min":0,"tarifa_base":0.0,"propina":0.0,"metodo_pago":"tarjeta","estado":"cancelado","calificacion_pasajero":null,"calificacion_conductor":null,"conductor_id":"DRV-975","pasajero_id":"PAX-1102","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.3","_batch_id":"BAT-20241103-ALL"},
  {"viaje_id":"VJ-M-0021","timestamp_inicio":"2024-11-03T14:00:00","timestamp_fin":"2024-11-03T14:45:00","ciudad":"Madrid","zona_origen":"Chamartín","zona_destino":"Barajas","distancia_km":12.0,"duracion_min":45,"tarifa_base":16.00,"propina":4.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-332","pasajero_id":"PAX-5512","tipo_vehiculo":"UberXL","surge_multiplier":1.0,"_sys_version":"v3.3","_batch_id":"BAT-20241103-ALL"},
  {"viaje_id":"VJ-B-0011","timestamp_inicio":"2024-11-03T16:30:00","timestamp_fin":"2024-11-03T17:00:00","ciudad":"Barcelona","zona_origen":"Gràcia","zona_destino":"Barceloneta","distancia_km":4.7,"duracion_min":30,"tarifa_base":7.50,"propina":1.50,"metodo_pago":"app_wallet","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":5,"conductor_id":"DRV-730","pasajero_id":"PAX-6631","tipo_vehiculo":"UberComfort","surge_multiplier":1.5,"_sys_version":"v3.3","_batch_id":"BAT-20241103-ALL"},
  {"viaje_id":"VJ-S-0011","timestamp_inicio":"2024-11-03T18:00:00","timestamp_fin":"2024-11-03T18:28:00","ciudad":"Sevilla","zona_origen":"Triana","zona_destino":"Macarena","distancia_km":5.5,"duracion_min":28,"tarifa_base":8.20,"propina":0.0,"metodo_pago":"efectivo","estado":"completado","calificacion_pasajero":3,"calificacion_conductor":4,"conductor_id":"DRV-561","pasajero_id":"PAX-8841","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.3","_batch_id":"BAT-20241103-ALL"}
]"""

val ruta7 = s"$rutaJSON/viajes_todas_ciudades_2024-11-03.json"
Files.write(Paths.get(ruta7), jsonMultiCiudad.getBytes(StandardCharsets.UTF_8))
println(s"✅ Creado: $ruta7")

✅ Creado: ./json_raw/viajes_todas_ciudades_2024-11-03.json


jsonMultiCiudad: String = """[
  {"viaje_id":"VJ-M-0020","timestamp_inicio":"2024-11-03T08:00:00","timestamp_fin":"2024-11-03T08:30:00","ciudad":"Madrid","zona_origen":"Sol","zona_destino":"Retiro","distancia_km":3.8,"duracion_min":30,"tarifa_base":6.20,"propina":1.00,"metodo_pago":"tarjeta","estado":"completado","calificacion_pasajero":5,"calificacion_conductor":5,"conductor_id":"DRV-441","pasajero_id":"PAX-2201","tipo_vehiculo":"UberX","surge_multiplier":1.0,"_sys_version":"v3.3","_batch_id":"BAT-20241103-ALL"},
  {"viaje_id":"VJ-B-0010","timestamp_inicio":"2024-11-03T09:15:00","timestamp_fin":"2024-11-03T09:50:00","ciudad":"Barcelona","zona_origen":"Eixample","zona_destino":"Sants","distancia_km":5.2,"duracion_min":35,"tarifa_base":8.80,"propina":0.0,"metodo_pago":"app_wallet","estado":"completado","calificacion_pasajero":4,"calificacion_conductor":4,"conductor_id":"DRV-610","pasajero_id":"PAX-4410","tipo_vehiculo":"UberX","surge_multiplier":1.1,"_sys_version":"v3.3","_batch_id":"BA

### Verificación de los 7 ficheros

In [10]:
val ficheros = new File(rutaJSON).listFiles().filter(_.getName.endsWith(".json")).sorted
println(s"Ficheros JSON en $rutaJSON:\n")
ficheros.foreach { f =>
  println(f"  ${f.getName}%-45s  ${f.length()} bytes")
}
println(s"\n✅ Total: ${ficheros.length} ficheros JSON listos")

Ficheros JSON en ./json_raw:

  viajes_barcelona_2024-11-01.json               3514 bytes
  viajes_bilbao_2024-11-01.json                  2502 bytes
  viajes_madrid_2024-11-01.json                  4938 bytes
  viajes_madrid_2024-11-02.json                  2490 bytes
  viajes_sevilla_2024-11-01.json                 2479 bytes
  viajes_todas_ciudades_2024-11-03.json          3982 bytes
  viajes_valencia_2024-11-01.json                3005 bytes

✅ Total: 7 ficheros JSON listos


ficheros: Array[File] = Array(
  .\json_raw\viajes_barcelona_2024-11-01.json,
  .\json_raw\viajes_bilbao_2024-11-01.json,
  .\json_raw\viajes_madrid_2024-11-01.json,
  .\json_raw\viajes_madrid_2024-11-02.json,
  .\json_raw\viajes_sevilla_2024-11-01.json,
  .\json_raw\viajes_todas_ciudades_2024-11-03.json,
  .\json_raw\viajes_valencia_2024-11-01.json
)

---

## 📂 Parte 4 — Leer todos los JSON

Spark lee la carpeta entera con un glob `*.json` y detecta el schema automáticamente desde una muestra. Como los ficheros son arrays JSON multi-línea, usamos `multiline = true`.

In [11]:
val dfRaw = spark.read
  .option("multiline", "true")   // los ficheros son arrays JSON formateados en varias líneas
  .json(s"$rutaJSON/*.json")

println("=== DataFrame RAW cargado desde JSON ===")
println(s"  Filas totales : ${dfRaw.count()}")
println(s"  Columnas      : ${dfRaw.columns.length}")
println()
println("Schema detectado automáticamente:")
dfRaw.printSchema()
println()
dfRaw.show(5, truncate = false)

=== DataFrame RAW cargado desde JSON ===
  Filas totales : 46
  Columnas      : 20

Schema detectado automáticamente:
root
 |-- _batch_id: string (nullable = true)
 |-- _sys_version: string (nullable = true)
 |-- calificacion_conductor: long (nullable = true)
 |-- calificacion_pasajero: long (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- conductor_id: string (nullable = true)
 |-- distancia_km: double (nullable = true)
 |-- duracion_min: long (nullable = true)
 |-- estado: string (nullable = true)
 |-- metodo_pago: string (nullable = true)
 |-- pasajero_id: string (nullable = true)
 |-- propina: double (nullable = true)
 |-- surge_multiplier: double (nullable = true)
 |-- tarifa_base: double (nullable = true)
 |-- timestamp_fin: string (nullable = true)
 |-- timestamp_inicio: string (nullable = true)
 |-- tipo_vehiculo: string (nullable = true)
 |-- viaje_id: string (nullable = true)
 |-- zona_destino: string (nullable = true)
 |-- zona_origen: string (nullable = true)




dfRaw: org.apache.spark.sql.package.DataFrame = [_batch_id: string, _sys_version: string ... 18 more fields]

> 💡 Observa que Spark ha detectado `_batch_id` y `_sys_version` — esos son los campos operacionales internos que descartaremos en la Parte 6.

---

## ⚙️ Parte 5 — Enriquecimiento con lógica de negocio

Añadimos columnas calculadas con **funciones nativas Spark** (más rápidas que UDFs porque Catalyst las puede analizar). Cada nueva columna responde una pregunta de BI.

In [12]:
val dfEnriquecido = dfRaw

  // 1. Parsear timestamps a tipo Timestamp
  .withColumn("ts_inicio",  to_timestamp(col("timestamp_inicio"), "yyyy-MM-dd'T'HH:mm:ss"))
  .withColumn("ts_fin",     to_timestamp(col("timestamp_fin"),    "yyyy-MM-dd'T'HH:mm:ss"))

  // 2. Extraer fecha y hora
  .withColumn("fecha",       to_date(col("ts_inicio")))
  .withColumn("hora_inicio", hour(col("ts_inicio")))

  // 3. Franja horaria (¿cuándo hay más demanda?)
  .withColumn("franja_horaria",
    when(col("hora_inicio").between(6, 9),    "Mañana pico")
    .when(col("hora_inicio").between(10, 13), "Mediodía")
    .when(col("hora_inicio").between(14, 17), "Tarde")
    .when(col("hora_inicio").between(18, 21), "Noche pico")
    .otherwise("Madrugada")
  )

  // 4. Importe total (tarifa + propina)
  .withColumn("importe_total",
    round(col("tarifa_base") + col("propina"), 2)
  )

  // 5. Importe con surge aplicado (explícito para BI)
  .withColumn("importe_con_surge",
    round(col("tarifa_base") * col("surge_multiplier"), 2)
  )

  // 6. Indicador viaje largo (>10 km)
  .withColumn("es_viaje_largo", col("distancia_km") > 10.0)

  // 7. Indicador viaje con propina
  .withColumn("tiene_propina", col("propina") > 0.0)

  // 8. Segmento de calidad del conductor
  .withColumn("segmento_conductor",
    when(col("estado") === "cancelado",        "Sin valorar")
    .when(col("calificacion_conductor") === 5, "Excelente")
    .when(col("calificacion_conductor") >= 4,  "Bueno")
    .when(col("calificacion_conductor") >= 3,  "Aceptable")
    .otherwise("Bajo")
  )

  // 9. Indicador surge activo
  .withColumn("surge_activo", col("surge_multiplier") > 1.0)

  // 10. Precio por km — protección de división por cero (cancelados)
  .withColumn("precio_por_km",
    when(col("distancia_km") > 0.0,
      round(col("tarifa_base") / col("distancia_km"), 2)
    ).otherwise(null)
  )

println("=== DataFrame enriquecido ===")
println(s"  Columnas originales         : ${dfRaw.columns.length}")
println(s"  Columnas tras enriquecimiento: ${dfEnriquecido.columns.length}")
println()
dfEnriquecido.select(
  "viaje_id", "ciudad", "fecha", "franja_horaria",
  "importe_total", "es_viaje_largo", "segmento_conductor", "surge_activo", "precio_por_km"
).show(8, truncate = false)

=== DataFrame enriquecido ===
  Columnas originales         : 20
  Columnas tras enriquecimiento: 32

+---------+------+----------+--------------+-------------+--------------+------------------+------------+-------------+
|viaje_id |ciudad|fecha     |franja_horaria|importe_total|es_viaje_largo|segmento_conductor|surge_activo|precio_por_km|
+---------+------+----------+--------------+-------------+--------------+------------------+------------+-------------+
|VJ-M-0001|Madrid|2024-11-01|Mañana pico   |7.6          |false         |Excelente         |false       |1.33         |
|VJ-M-0002|Madrid|2024-11-01|Mañana pico   |5.1          |false         |Bueno             |true        |1.59         |
|VJ-M-0003|Madrid|2024-11-01|Mañana pico   |0.0          |false         |Sin valorar       |false       |NULL         |
|VJ-M-0004|Madrid|2024-11-01|Mañana pico   |11.2         |false         |Excelente         |true        |1.51         |
|VJ-M-0005|Madrid|2024-11-01|Mediodía      |5.3          |

dfEnriquecido: org.apache.spark.sql.package.DataFrame = [_batch_id: string, _sys_version: string ... 30 more fields]

---

## ✂️ Parte 6 — Selección de columnas para el equipo de BI

Excluimos campos operacionales (`_batch_id`, `_sys_version`), strings raw de timestamp y los `Timestamp` parseados (BI trabaja con `fecha` + `hora_inicio`). Regla: **solo lo que responde preguntas de negocio**.

In [13]:
val dfBI = dfEnriquecido.select(
  // Identificadores temporales
  col("viaje_id"),
  col("fecha"),
  col("hora_inicio"),
  col("franja_horaria"),

  // Geografía
  col("ciudad"),
  col("zona_origen"),
  col("zona_destino"),

  // Operación
  col("tipo_vehiculo"),
  col("estado"),
  col("distancia_km"),
  col("duracion_min"),
  col("es_viaje_largo"),

  // Economía
  col("tarifa_base"),
  col("propina"),
  col("importe_total"),
  col("importe_con_surge"),
  col("surge_multiplier"),
  col("surge_activo"),
  col("tiene_propina"),
  col("precio_por_km"),
  col("metodo_pago"),

  // Calidad
  col("calificacion_pasajero"),
  col("calificacion_conductor"),
  col("segmento_conductor"),

  // Actores (anonimizados — solo IDs)
  col("conductor_id"),
  col("pasajero_id")

  // ❌ Excluidos intencionadamente:
  //   _batch_id, _sys_version → campos internos del sistema operacional
  //   timestamp_inicio / timestamp_fin (String raw) → reemplazados por fecha + hora_inicio
  //   ts_inicio, ts_fin → BI no los necesita
)

println("=== DataFrame BI final ===")
println(s"  Columnas seleccionadas: ${dfBI.columns.length}")
println(s"  Filas: ${dfBI.count()}")
println()
println("Columnas del DataFrame BI:")
dfBI.columns.zipWithIndex.foreach { case (c, i) =>
  println(f"  ${i + 1}%2d. $c")
}
println()
dfBI.show(5, truncate = true)

=== DataFrame BI final ===
  Columnas seleccionadas: 26
  Filas: 46

Columnas del DataFrame BI:
   1. viaje_id
   2. fecha
   3. hora_inicio
   4. franja_horaria
   5. ciudad
   6. zona_origen
   7. zona_destino
   8. tipo_vehiculo
   9. estado
  10. distancia_km
  11. duracion_min
  12. es_viaje_largo
  13. tarifa_base
  14. propina
  15. importe_total
  16. importe_con_surge
  17. surge_multiplier
  18. surge_activo
  19. tiene_propina
  20. precio_por_km
  21. metodo_pago
  22. calificacion_pasajero
  23. calificacion_conductor
  24. segmento_conductor
  25. conductor_id
  26. pasajero_id

+---------+----------+-----------+--------------+------+-----------+------------+-------------+----------+------------+------------+--------------+-----------+-------+-------------+-----------------+----------------+------------+-------------+-------------+-----------+---------------------+----------------------+------------------+------------+-----------+
| viaje_id|     fecha|hora_inicio|franja_

dfBI: org.apache.spark.sql.package.DataFrame = [viaje_id: string, fecha: date ... 24 more fields]

---

## 💾 Parte 7 — Escribir Parquet particionado por `ciudad`

Particionar por `ciudad` permite **partition pruning**: cuando BI filtre por una ciudad, Spark solo leerá esa subcarpeta.

In [14]:
val rutaParquetBI = s"$rutaSalida/parquet_bi"

val inicio = System.nanoTime()

// repartition("ciudad") agrupa cada ciudad en una partición Spark → un único writer por task
// (mismo patrón que aplicamos en el Caso 2 para evitar OutOfMemory en escrituras particionadas).
dfBI
  .repartition(col("ciudad"))
  .write
  .mode("overwrite")
  .partitionBy("ciudad")
  .parquet(rutaParquetBI)

val tiempoMs = (System.nanoTime() - inicio) / 1000000L

println(s"✅ Parquet BI escrito en $tiempoMs ms")
println(s"   Ruta: $rutaParquetBI")
println()
println("Particiones generadas:")
new File(rutaParquetBI)
  .listFiles()
  .filter(_.isDirectory)
  .map(_.getName)
  .sorted
  .foreach(p => println(s"  $p/"))

✅ Parquet BI escrito en 9568 ms
   Ruta: ./salida/parquet_bi

Particiones generadas:
  ciudad=Barcelona/
  ciudad=Bilbao/
  ciudad=Madrid/
  ciudad=Sevilla/
  ciudad=Valencia/


rutaParquetBI: String = "./salida/parquet_bi"
inicio: Long = 9384943414100L
tiempoMs: Long = 9568L

---

## 🔍 Parte 8 — Leer el Parquet y crear la vista SQL

In [15]:
val dfParquetBI = spark.read.parquet(rutaParquetBI)

dfParquetBI.createOrReplaceTempView("viajes_rapidx")

println("✅ Vista 'viajes_rapidx' registrada en Spark SQL")
println(s"   Filas en la vista: ${dfParquetBI.count()}")
println()
println("Schema de la vista:")
dfParquetBI.printSchema()

✅ Vista 'viajes_rapidx' registrada en Spark SQL
   Filas en la vista: 46

Schema de la vista:
root
 |-- viaje_id: string (nullable = true)
 |-- fecha: date (nullable = true)
 |-- hora_inicio: integer (nullable = true)
 |-- franja_horaria: string (nullable = true)
 |-- zona_origen: string (nullable = true)
 |-- zona_destino: string (nullable = true)
 |-- tipo_vehiculo: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- distancia_km: double (nullable = true)
 |-- duracion_min: long (nullable = true)
 |-- es_viaje_largo: boolean (nullable = true)
 |-- tarifa_base: double (nullable = true)
 |-- propina: double (nullable = true)
 |-- importe_total: double (nullable = true)
 |-- importe_con_surge: double (nullable = true)
 |-- surge_multiplier: double (nullable = true)
 |-- surge_activo: boolean (nullable = true)
 |-- tiene_propina: boolean (nullable = true)
 |-- precio_por_km: double (nullable = true)
 |-- metodo_pago: string (nullable = true)
 |-- calificacion_pasajero: lo

dfParquetBI: org.apache.spark.sql.package.DataFrame = [viaje_id: string, fecha: date ... 24 more fields]

---

## 📊 Parte 9 — Consultas de negocio del equipo de BI

A partir de aquí trabajamos **exclusivamente con SQL**.

### BI-1 — Ingresos totales y viajes por ciudad

In [16]:
spark.sql("""
  SELECT
    ciudad,
    COUNT(*)                                                AS total_viajes,
    SUM(CASE WHEN estado = 'completado' THEN 1 ELSE 0 END)  AS viajes_completados,
    SUM(CASE WHEN estado = 'cancelado'  THEN 1 ELSE 0 END)  AS viajes_cancelados,
    ROUND(SUM(importe_total), 2)                            AS ingresos_totales,
    ROUND(AVG(importe_total), 2)                            AS ticket_medio,
    ROUND(AVG(distancia_km), 1)                             AS distancia_media_km
  FROM viajes_rapidx
  GROUP BY ciudad
  ORDER BY ingresos_totales DESC
""").show(truncate = false)

+---------+------------+------------------+-----------------+----------------+------------+------------------+
|ciudad   |total_viajes|viajes_completados|viajes_cancelados|ingresos_totales|ticket_medio|distancia_media_km|
+---------+------------+------------------+-----------------+----------------+------------+------------------+
|Madrid   |17          |15                |2                |169.0           |9.94        |5.9               |
|Barcelona|9           |8                 |1                |89.0            |9.89        |5.6               |
|Valencia |7           |6                 |1                |70.7            |10.1        |6.0               |
|Sevilla  |7           |6                 |1                |50.5            |7.21        |4.6               |
|Bilbao   |6           |4                 |2                |40.3            |6.72        |3.7               |
+---------+------------+------------------+-----------------+----------------+------------+------------------+



### BI-2 — Demanda por franja horaria

In [17]:
spark.sql("""
  SELECT
    franja_horaria,
    COUNT(*)                                                AS total_viajes,
    ROUND(SUM(importe_total), 2)                            AS ingresos,
    ROUND(AVG(surge_multiplier), 2)                         AS surge_medio,
    SUM(CASE WHEN surge_activo = true THEN 1 ELSE 0 END)    AS viajes_con_surge
  FROM viajes_rapidx
  WHERE estado = 'completado'
  GROUP BY franja_horaria
  ORDER BY total_viajes DESC
""").show(truncate = false)

+--------------+------------+--------+-----------+----------------+
|franja_horaria|total_viajes|ingresos|surge_medio|viajes_con_surge|
+--------------+------------+--------+-----------+----------------+
|Mañana pico   |15          |154.4   |1.05       |3               |
|Mediodía      |10          |118.5   |1.03       |1               |
|Tarde         |8           |90.9    |1.3        |5               |
|Noche pico    |5           |46.4    |1.44       |3               |
|Madrugada     |1           |9.3     |1.8        |1               |
+--------------+------------+--------+-----------+----------------+



### BI-3 — Rendimiento por tipo de vehículo

In [18]:
spark.sql("""
  SELECT
    tipo_vehiculo,
    COUNT(*)                              AS total_viajes,
    ROUND(AVG(distancia_km), 1)           AS distancia_media_km,
    ROUND(AVG(importe_total), 2)          AS ingreso_medio,
    ROUND(AVG(precio_por_km), 2)          AS precio_medio_km,
    ROUND(AVG(calificacion_pasajero), 2)  AS nota_media_pasajero
  FROM viajes_rapidx
  WHERE estado = 'completado'
  GROUP BY tipo_vehiculo
  ORDER BY ingreso_medio DESC
""").show(truncate = false)

+-------------+------------+------------------+-------------+---------------+-------------------+
|tipo_vehiculo|total_viajes|distancia_media_km|ingreso_medio|precio_medio_km|nota_media_pasajero|
+-------------+------------+------------------+-------------+---------------+-------------------+
|UberXL       |8           |11.9              |19.39        |1.34           |4.63               |
|UberComfort  |8           |6.5               |11.4         |1.52           |4.75               |
|UberX        |23          |4.3               |7.53         |1.65           |4.13               |
+-------------+------------+------------------+-------------+---------------+-------------------+



### BI-4 — Top 5 conductores por ingresos generados

In [19]:
spark.sql("""
  SELECT
    conductor_id,
    COUNT(*)                              AS viajes_completados,
    ROUND(SUM(importe_total), 2)          AS ingresos_generados,
    ROUND(AVG(calificacion_pasajero), 2)  AS nota_media,
    segmento_conductor
  FROM viajes_rapidx
  WHERE estado = 'completado'
  GROUP BY conductor_id, segmento_conductor
  ORDER BY ingresos_generados DESC
  LIMIT 5
""").show(truncate = false)

+------------+------------------+------------------+----------+------------------+
|conductor_id|viajes_completados|ingresos_generados|nota_media|segmento_conductor|
+------------+------------------+------------------+----------+------------------+
|DRV-441     |4                 |41.4              |4.75      |Excelente         |
|DRV-119     |2                 |37.0              |5.0       |Excelente         |
|DRV-803     |2                 |35.3              |4.5       |Excelente         |
|DRV-207     |3                 |29.4              |4.33      |Bueno             |
|DRV-975     |2                 |27.9              |4.5       |Excelente         |
+------------+------------------+------------------+----------+------------------+



### BI-5 — Análisis de métodos de pago y propinas

In [20]:
spark.sql("""
  SELECT
    metodo_pago,
    COUNT(*)                                                  AS total_viajes,
    SUM(CASE WHEN tiene_propina = true THEN 1 ELSE 0 END)     AS viajes_con_propina,
    ROUND(
      100.0 * SUM(CASE WHEN tiene_propina = true THEN 1 ELSE 0 END)
            / COUNT(*), 1
    )                                                          AS pct_con_propina,
    ROUND(AVG(CASE WHEN tiene_propina = true THEN propina END), 2) AS propina_media
  FROM viajes_rapidx
  WHERE estado = 'completado'
  GROUP BY metodo_pago
  ORDER BY total_viajes DESC
""").show(truncate = false)

+-----------+------------+------------------+---------------+-------------+
|metodo_pago|total_viajes|viajes_con_propina|pct_con_propina|propina_media|
+-----------+------------+------------------+---------------+-------------+
|tarjeta    |22          |20                |90.9           |2.21         |
|efectivo   |11          |3                 |27.3           |1.0          |
|app_wallet |6           |4                 |66.7           |1.25         |
+-----------+------------+------------------+---------------+-------------+



### BI-6 — Impacto del surge por ciudad

In [21]:
spark.sql("""
  SELECT
    ciudad,
    SUM(CASE WHEN surge_activo = true  THEN 1 ELSE 0 END)               AS viajes_con_surge,
    SUM(CASE WHEN surge_activo = false THEN 1 ELSE 0 END)               AS viajes_sin_surge,
    ROUND(AVG(CASE WHEN surge_activo = true THEN surge_multiplier END), 2) AS surge_medio_activo,
    ROUND(SUM(CASE WHEN surge_activo = true THEN importe_con_surge ELSE 0 END), 2) AS ingresos_extra_surge
  FROM viajes_rapidx
  WHERE estado = 'completado'
  GROUP BY ciudad
  ORDER BY ingresos_extra_surge DESC
""").show(truncate = false)

+---------+----------------+----------------+------------------+--------------------+
|ciudad   |viajes_con_surge|viajes_sin_surge|surge_medio_activo|ingresos_extra_surge|
+---------+----------------+----------------+------------------+--------------------+
|Madrid   |6               |9               |1.62              |94.19               |
|Barcelona|3               |5               |1.4               |37.25               |
|Valencia |2               |4               |1.45              |24.32               |
|Sevilla  |1               |5               |1.4               |10.92               |
|Bilbao   |1               |3               |1.3               |10.53               |
+---------+----------------+----------------+------------------+--------------------+



### BI-7 — Tasa de cancelación y rutas más canceladas

In [22]:
spark.sql("""
  SELECT
    ciudad,
    zona_origen,
    COUNT(*)                                                AS total_intentos,
    SUM(CASE WHEN estado = 'cancelado' THEN 1 ELSE 0 END)   AS cancelados,
    ROUND(
      100.0 * SUM(CASE WHEN estado = 'cancelado' THEN 1 ELSE 0 END)
            / COUNT(*), 1
    )                                                        AS tasa_cancelacion_pct
  FROM viajes_rapidx
  GROUP BY ciudad, zona_origen
  HAVING COUNT(*) >= 1
  ORDER BY tasa_cancelacion_pct DESC, total_intentos DESC
  LIMIT 10
""").show(truncate = false)

+---------+---------------+--------------+----------+--------------------+
|ciudad   |zona_origen    |total_intentos|cancelados|tasa_cancelacion_pct|
+---------+---------------+--------------+----------+--------------------+
|Madrid   |Moncloa        |1             |1         |100.0               |
|Madrid   |Vallecas       |1             |1         |100.0               |
|Barcelona|Sagrada Familia|1             |1         |100.0               |
|Bilbao   |Centro         |1             |1         |100.0               |
|Sevilla  |Nervión        |2             |1         |50.0                |
|Valencia |Centro         |2             |1         |50.0                |
|Bilbao   |Guggenheim     |2             |1         |50.0                |
|Sevilla  |Triana         |3             |0         |0.0                 |
|Barcelona|Sants          |2             |0         |0.0                 |
|Madrid   |Salamanca      |2             |0         |0.0                 |
+---------+--------------

---

## 📝 Parte 10 — Preguntas de reflexión (respuestas)

### 🅰️ Sobre el pipeline

**1. ¿Por qué leemos los JSON con `spark.read.json("carpeta/*.json")` en lugar de uno a uno?**

Porque Spark **paraleliza** la lectura: cada fichero se asigna a una task distinta y se procesa en paralelo. Además unifica los schemas automáticamente (si un fichero tiene una columna que otros no, aparece como `null` en los demás). Leer uno a uno y unirlos a mano sería más lento, más código y más frágil.

**2. ¿Qué columnas del JSON original excluiste del DataFrame BI y por qué?**

- `_batch_id`, `_sys_version` → metadatos del sistema operacional, **no son negocio**.
- `timestamp_inicio`, `timestamp_fin` (String raw) → reemplazados por `fecha` + `hora_inicio` (más útil para SQL).
- `ts_inicio`, `ts_fin` (Timestamp) → BI prefiere trabajar con `fecha` y `hora_inicio` por separado para agrupar y filtrar.

**3. ¿Diferencia técnica entre `inferSchema` en CSV y la inferencia automática de JSON?**

En CSV, `inferSchema = true` obliga a Spark a hacer **dos pasadas** (una para inferir, otra para cargar). En JSON, el schema viene **embebido en cada objeto** (los tipos están implícitos en la sintaxis: `"abc"` es String, `123` es Long, `1.5` es Double, `true` es Boolean, `null` es nullable). Spark deduce el schema en una sola pasada inspeccionando una muestra de los objetos.

### 🅱️ Sobre las funciones de Spark

**1. ¿Por qué `when / otherwise` es mejor que una UDF para `franja_horaria` y `segmento_conductor`?**

Porque `when / otherwise` es una **expresión Catalyst nativa**: el optimizador la entiende, puede aplicar predicate pushdown, fusionar con otros filtros, paralelizar y compilar a bytecode optimizado (Tungsten). Una UDF es una **caja negra** para Catalyst: no la puede inspeccionar ni reordenar, y cada llamada implica serialización/deserialización entre la JVM y el ejecutor.

**2. ¿Qué pasaría si no protegieras la división por cero en `precio_por_km`?**

Para los viajes cancelados (`distancia_km = 0.0`), `tarifa_base / 0.0` daría:
- En Spark con `Double`: el resultado es `Infinity` o `NaN` (no lanza excepción), pero **rompe agregaciones** posteriores: `AVG(precio_por_km)` devolvería `NaN` y propagaría a otros KPIs.
- Cualquier consulta BI que ordene por ese campo mostraría los `Infinity` arriba, ensuciando el resultado.

**3. ¿Diferencia entre `to_timestamp` y `to_date`?**

- `to_timestamp` → parsea String a `TimestampType` (fecha **+ hora con segundos/ms**). Lo usamos en `ts_inicio` para luego extraer `hour()`.
- `to_date` → parsea (o convierte un Timestamp) a `DateType` (solo año-mes-día, **sin hora**). Lo usamos para `fecha`, perfecta para agrupar BI por día.

### 🅲 Sobre el Parquet particionado

**1. ¿Qué consulta BI se beneficia más del particionado por `ciudad`?**

Cualquier consulta con `WHERE ciudad = 'Madrid'`. La **BI-1** (agrupada por ciudad) tendría que leer todo el dataset igualmente, pero las consultas operacionales tipo *"top conductores en Madrid"* solo leen `ciudad=Madrid/` gracias al **partition pruning**.

**2. ¿Añadirías `fecha` como segunda partición? Riesgos.**

Sí ayudaría a queries con filtro `ciudad + fecha`, **pero**: con 5 ciudades × 30 días = 150 carpetas para muy pocos viajes (< 50 totales en este ejemplo). Riesgo del **small files problem**: muchísimos ficheros Parquet diminutos, planificación más cara que la lectura. En producción real, si hay miles de viajes/día, particionar por `(ciudad, anio_mes)` sí sería razonable; nunca por `(ciudad, fecha_completa)` salvo volúmenes muy altos.

### 🅳 Sobre Spark SQL

**1. ¿Por qué `100.0` en lugar de `100` en BI-5?**

Si escribes `100 * SUM(...) / COUNT(*)` con dos enteros, Spark hace **división entera** y trunca decimales: `3 / 7 = 0`. Con `100.0` fuerzas a que toda la expresión se evalúe en `Double` y obtienes el porcentaje real (`42.86`).

**2. ¿Cómo actualizarías la vista si llegan los JSON de diciembre?**

Dos opciones:
- **Reprocesar todo:** copiar los nuevos JSON a `json_raw/` y re-ejecutar el pipeline desde la Parte 4 con `mode("overwrite")`.
- **Append incremental:** leer solo los nuevos JSON, aplicar el mismo enriquecimiento + selección, y escribir con `mode("append").partitionBy("ciudad")`. Spark añade los datos a las subcarpetas existentes sin tocar las anteriores. Después basta `spark.read.parquet(rutaParquetBI).createOrReplaceTempView("viajes_rapidx")` para refrescar la vista.

---

## 🗺️ Flujo completo del pipeline

```text
1. 7 ficheros JSON sintéticos
   (Madrid×2, Barcelona, Sevilla, Valencia, Bilbao, multi-ciudad)
         │
         ▼
2. spark.read.json("json_raw/*.json")
   → DataFrame raw (51 filas, 20 columnas, incluye campos _sys)
         │
         ▼
3. Enriquecimiento (10 columnas nuevas)
   → ts_inicio, fecha, franja_horaria, importe_total, etc.
         │
         ▼
4. Selección BI (descarta _sys_*, timestamps raw, ts_*)
   → DataFrame final (26 columnas analíticas)
         │  partitionBy("ciudad")
         ▼
5. Parquet particionado (Data Lake)
   ciudad=Madrid/  ciudad=Barcelona/  ciudad=Sevilla/ ...
         │  createOrReplaceTempView
         ▼
6. Vista SQL viajes_rapidx
         │
         ▼
7. Consultas BI (BI-1 a BI-7)
```

## 💡 Para recordar

> Un pipeline JSON → Parquet → BI no es solo una conversión de formato: es **eliminar el ruido operacional**, **enriquecer con lógica de negocio** y **organizar físicamente los datos** (particionado) para que las consultas analíticas sean rápidas y baratas.

---

## 🛑 Cierre

In [24]:
spark.stop()
println("✅ Caso Práctico 3 — RapidX completado")

✅ Caso Práctico 3 — RapidX completado
